# Threat Context Engineering v0.3 — AI Knowledge Graph

**Colab profile:** GLiNER ON by default · Qwen ON by default · ATT&CK + D3FEND + Attack Flow + STIX/OpenCTI.

AI outputs are analyst-review candidates. The notebook never permits GLiNER or Qwen to silently rewrite source evidence.

In [ ]:
import os, platform, subprocess, sys
os.environ.setdefault('TCE_AI_GLINER', '1')
os.environ.setdefault('TCE_AI_QWEN', '1')
os.environ.setdefault('TCE_QWEN_MODEL', 'auto')
os.environ.setdefault('TCE_QWEN_4BIT', '1')
print('GLiNER:', os.environ['TCE_AI_GLINER'], '| Qwen:', os.environ['TCE_AI_QWEN'])
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
import importlib, os, pathlib, site, subprocess, sys
REPO = 'https://github.com/Ridd1kulusC0d3r/Cyber-Threat-Context-Modeling.git'
BRANCH = os.environ.get('TCE_BRANCH', 'main')
ROOT = pathlib.Path('/content/Cyber-Threat-Context-Modeling')

# Colab keeps one Python kernel alive. An editable install can create a .pth file
# that is only processed on interpreter startup, so we also add src/ explicitly.
if ROOT.exists() and (ROOT / '.git').exists():
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', BRANCH, '--depth', '1'], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', '.[colab]'], check=True)

SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
site.addsitedir(str(SRC))
importlib.invalidate_caches()

import tce
print('Installed from:', ROOT)
print('TCE import OK:', tce.__version__)
print('TCE module:', tce.__file__)
subprocess.run([sys.executable, '-m', 'pip', 'show', 'tce-toolkit'], check=False)


### Colab live-kernel note

The setup cell deliberately injects the repository `src/` directory into the running kernel after the editable install. This avoids a Colab/Jupyter edge case where a newly installed editable package may not become importable until the interpreter restarts. Re-running setup also refreshes the existing clone to the selected branch instead of silently using stale code.

## TCE Web Workbench — porta 3000

Este frontend leve roda **na própria VM do Colab**, lê o mesmo TCE Case e expõe Visão Geral, Cenários, Inteligência, ATT&CK/D3FEND, IA e Grafo. O link gerado usa o proxy seguro do Colab para a porta 3000.

In [ ]:
import os, pathlib, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

PORT = 3000
CASE_DIR = (ROOT / 'examples/cases/enterprise-identity').resolve()
os.environ['TCE_CASE_DIR'] = str(CASE_DIR)

# Re-running this cell replaces the previous local server instead of stacking processes.
if 'tce_ui_proc' in globals() and tce_ui_proc.poll() is None:
    tce_ui_proc.terminate()
    try:
        tce_ui_proc.wait(timeout=3)
    except subprocess.TimeoutExpired:
        tce_ui_proc.kill()

env = os.environ.copy()
tce_ui_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'tce.webapp:app', '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=str(ROOT),
    env=env,
)

ready = False
for _ in range(30):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=1) as response:
            ready = response.status == 200
            if ready:
                break
    except Exception:
        time.sleep(1)

if not ready:
    raise RuntimeError('TCE UI did not become ready on port 3000')

try:
    from google.colab import output
    proxy_url = output.eval_js(f'google.colab.kernel.proxyPort({PORT})')
    display(HTML(f'<a href="{proxy_url}" target="_blank" style="display:inline-block;padding:12px 18px;background:#23d1b5;color:#03100e;border-radius:10px;font-weight:700;text-decoration:none">Open TCE Workbench</a>'))
    print('TCE UI:', proxy_url)
except Exception as exc:
    print(f'TCE UI ready on port {PORT}. Colab proxy link could not be generated automatically:', exc)


In [ ]:
from tce.ai.config import AIConfig
from tce.ai.copilot import TCECopilot
config = AIConfig()
print(config)
copilot = TCECopilot(config)
print('Loading GLiNER...')
copilot.gliner.load()
print('Loading Qwen:', config.qwen_model)
copilot.qwen.load()
print('AI READY')


In [ ]:
from tce.standards.attack import sync_attack, AttackIndex
from tce.standards.d3fend import sync_d3fend, D3FENDIndex
print('ATT&CK cache:', sync_attack())
print('D3FEND cache:', sync_d3fend())
attack = AttackIndex.load()
d3fend = D3FENDIndex.load()
print('ATT&CK techniques indexed:', len(attack.techniques))


## AI Evidence Packet

Replace SOURCE_TEXT with a threat report excerpt, architecture note, or other authorized defensive analysis input. GLiNER extracts candidates; Qwen proposes analytical links. The original text remains unchanged.

In [ ]:
SOURCE_TEXT = '''An internal review found that a privileged SaaS administration role can modify enterprise identity settings. Existing audit telemetry records the administrator and source address, but the target-resource field is not consistently populated. The security team wants to determine whether this trust relationship creates a high-priority identity-control scenario and what telemetry is required to validate detection.'''
result = copilot.analyze_text(SOURCE_TEXT, source_id='COLAB-SOURCE-001', case_id='CASE-001')
print('Packet:', result['evidence_packet']['packet_id'])
print('Entity candidates:', len(result['entity_candidates']))
print('Integrity:', result['integrity'])
result


In [ ]:
from tce.io import load_case
from tce.knowledge_graph import build_operational_graph, graph_summary, write_graph
case_dir = 'examples/cases/enterprise-identity'
entities, errors = load_case(case_dir)
assert not errors, errors
kg = build_operational_graph(entities, attack_index=attack, d3fend_index=d3fend)
print(graph_summary(kg))
write_graph(kg, '/content/tce-kg.json', 'json')
write_graph(kg, '/content/tce-kg.ttl', 'ttl')


In [ ]:
from pyvis.network import Network
from IPython.display import HTML, display
net = Network(height='700px', width='100%', directed=True, notebook=True, cdn_resources='in_line')
for node, attrs in kg.nodes(data=True):
    net.add_node(node, label=attrs.get('label', node), title=attrs.get('kind', 'entity'))
for source, target, attrs in kg.edges(data=True):
    net.add_edge(source, target, title=attrs.get('relation', 'related'))
net.write_html('/content/tce-kg.html', notebook=True)
display(HTML(filename='/content/tce-kg.html'))


In [ ]:
from tce.standards.attack_flow import scenario_to_attack_flow, write_attack_flow
from tce.standards.navigator import export_navigator_layer, write_navigator
from tce.standards.stix_export import export_case_stix, write_stix
scenario = entities['TS-001']['data']
write_attack_flow(scenario_to_attack_flow(scenario, attack), '/content/tce-attack-flow.json')
write_navigator(export_navigator_layer(entities), '/content/tce-navigator.json')
write_stix(export_case_stix(entities), '/content/tce-case.stix.json')
print('Exports ready in /content')


## Optional OpenCTI import

No upload occurs automatically. First inspect the generated STIX bundle. Only run an import after setting OPENCTI_URL and OPENCTI_TOKEN and deciding to transfer the reviewed data.

In [ ]:
import os
from tce.standards.opencti import bundle_summary
bundle_path = '/content/tce-case.stix.json'
print(bundle_summary(bundle_path))
print('OPENCTI configured:', bool(os.getenv('OPENCTI_URL') and os.getenv('OPENCTI_TOKEN')))
print('Use tce opencti-push /content/tce-case.stix.json --commit only after review.')
